# 1. Phase 2 — Efficient Data Loading & Memory Management

Prototype pipeline for **one representative daily file** only.

Goals: chunked reading, dtype control, country-code investigation, Internet aggregation options, and a baseline vs optimized memory comparison.

**Out of scope:** full 62-day processing, EDA, area selection, and models.

## 1.1 Configuration

In [ ]:
from __future__ import annotations

import gc
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

# Repository root (notebooks/ -> project root)
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data.loading import (
    COLUMN_NAMES,
    INTERNET_TASK_COLUMNS,
    INTERNET_TASK_DTYPES,
    add_timestamp_column,
    discover_zip_archives,
    extract_zip_member,
    find_member,
    get_process_rss_mb,
    read_traffic_file,
    remove_path,
    validate_float32_precision,
)
from src.data.preprocessing import (
    aggregate_internet_traffic,
    merge_chunk_aggregates,
    summarize_country_internet,
)

# Configure dataset location without hard-coding a personal Drive path.
# Priority: MILAN_DATA_DIR / DATA_DIR env → DATA_DIR cell override below.
DATA_DIR = Path(os.environ.get("MILAN_DATA_DIR") or os.environ.get("DATA_DIR") or r"C:\Users\user\Downloads")
REPRESENTATIVE_MEMBER = "sms-call-internet-mi-2013-11-01.txt"
CHUNKSIZE = 500_000
TMP_DIR = ROOT / "tmp" / "phase2_extract"
SAMPLES_DIR = ROOT / "data" / "samples"
METRICS_DIR = ROOT / "results" / "metrics"

TMP_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("CHUNKSIZE:", CHUNKSIZE)
print("Columns:", COLUMN_NAMES)

## 1.2 Select Representative File

Prefer `sms-call-internet-mi-2013-11-01.txt`. Extract **only** that member from its ZIP, then delete the temporary TXT after processing.

In [ ]:
zip_paths = discover_zip_archives(DATA_DIR)
print(f"Found {len(zip_paths)} ZIP archive(s) under DATA_DIR")
for zp in zip_paths:
    print(" -", zp.name, f"({zp.stat().st_size / 1e9:.3f} GB)")

member = find_member(REPRESENTATIVE_MEMBER, zip_paths=zip_paths)
print("Selected member:", member.member_name)
print("Archive:", member.zip_path.name)
print("Uncompressed size (MB):", round(member.uncompressed_size / 1e6, 2))

txt_path = extract_zip_member(member.zip_path, member.member_name, TMP_DIR)
print("Temporarily extracted to:", txt_path)

## 1.3 Inspect Country Codes

Official docs: Internet activity is associated with the phone **country code** of the users generating CDRs in a square/interval. We quantify how traffic is distributed before choosing an aggregation rule.

In [ ]:
country_parts = []
n_rows = 0
n_missing_cc = 0
unique_squares = set()
unique_ts = set()

for chunk in read_traffic_file(
    txt_path,
    chunksize=CHUNKSIZE,
    usecols=INTERNET_TASK_COLUMNS,
    dtype=INTERNET_TASK_DTYPES,
):
    n_rows += len(chunk)
    n_missing_cc += int(chunk["country_code"].isna().sum())
    unique_squares.update(chunk["square_id"].tolist())
    unique_ts.update(chunk["time_interval"].tolist())
    country_parts.append(summarize_country_internet(chunk))

cc_all = pd.concat(country_parts, ignore_index=True)
cc_summary = (
    cc_all.groupby("country_code", dropna=False)
    .agg(
        n_rows=("n_rows", "sum"),
        n_internet_non_null=("n_internet_non_null", "sum"),
        internet_sum=("internet_sum", "sum"),
    )
    .reset_index()
    .sort_values("internet_sum", ascending=False)
)
total_internet = float(cc_summary["internet_sum"].sum())
cc_summary["internet_share"] = cc_summary["internet_sum"] / total_internet

ts_sorted = sorted(unique_ts)
diffs = np.diff(ts_sorted)

print("Raw rows:", n_rows)
print("Unique squares:", len(unique_squares))
print("Unique timestamps:", len(unique_ts))
print("Timestamp range (UTC):", pd.to_datetime(min(unique_ts), unit="ms", utc=True), "→", pd.to_datetime(max(unique_ts), unit="ms", utc=True))
print("All global unique timestamps 10 minutes apart:", bool((diffs == 600_000).all()))
print("Missing country_code values:", n_missing_cc)
print("Unique country codes:", cc_summary["country_code"].nunique())
print("Internet share country 39:", float(cc_summary.loc[cc_summary["country_code"] == 39, "internet_share"].sum()))
display(cc_summary.head(15))

### Preliminary aggregation recommendation

- Documentation treats rows as square × time × **country_code** activity.
- On 2013-11-01, country **39 (Italy)** accounts for ~**99.72%** of Internet traffic sum; other codes are real but small (~0.28%).
- **Recommended default:** `sum_all_countries` — total Internet activity in a geographic cell is the sum over country codes.
- `country_39_only` remains a useful sensitivity check (nearly identical totals) but discards documented non-39 activity without a research justification.

## 1.4 Baseline Loading

Full-day load with pandas default dtypes (no chunking). Used only for one file.

In [ ]:
gc.collect()
mem_before_baseline = get_process_rss_mb()
t0 = time.perf_counter()
df_base = read_traffic_file(txt_path, chunksize=None, usecols=None, dtype=None)
baseline_seconds = time.perf_counter() - t0
mem_after_baseline = get_process_rss_mb()
df_base_mem_mb = df_base.memory_usage(deep=True).sum() / (1024 * 1024)

print("Baseline seconds:", round(baseline_seconds, 3))
print("RSS before (MB):", round(mem_before_baseline, 2))
print("RSS after (MB):", round(mem_after_baseline, 2))
print("RSS delta (MB):", round(mem_after_baseline - mem_before_baseline, 2))
print("DataFrame memory (MB):", round(df_base_mem_mb, 2))
print(df_base.dtypes)

df_base_internet = df_base[list(INTERNET_TASK_COLUMNS)].copy()
agg_sum_base = aggregate_internet_traffic(df_base_internet, strategy="sum_all_countries")
agg_39_base = aggregate_internet_traffic(df_base_internet, strategy="country_39_only")
base_total_sum = float(df_base_internet["internet_traffic"].sum(skipna=True))
base_total_39 = float(
    df_base_internet.loc[df_base_internet["country_code"] == 39, "internet_traffic"].sum(skipna=True)
)
spot_keys = list(
    agg_sum_base.head(5)[["square_id", "time_interval", "internet_traffic"]].itertuples(index=False, name=None)
)

del df_base, df_base_internet
gc.collect()
print("RSS after deleting baseline DataFrame (MB):", round(get_process_rss_mb(), 2))

## 1.5 Optimized Chunked Loading

- `chunksize=500_000`
- `usecols` limited to Internet-task columns
- Explicit dtypes (`int32` / `int64` / `float32`)
- Incremental aggregation per chunk, then merge

In [ ]:
# float32 precision check on first chunk (vs float64 default parse)
probe_f64 = next(
    read_traffic_file(txt_path, chunksize=100_000, usecols=INTERNET_TASK_COLUMNS, dtype=None)
)
precision = validate_float32_precision(probe_f64["internet_traffic"])
print("float32 precision check:", precision)
del probe_f64
gc.collect()

mem_before_opt = get_process_rss_mb()
peak_opt = mem_before_opt
t0 = time.perf_counter()
chunk_aggs_sum = []
chunk_aggs_39 = []
opt_rows = 0
opt_total_sum = 0.0
opt_total_39 = 0.0

for chunk in read_traffic_file(
    txt_path,
    chunksize=CHUNKSIZE,
    usecols=INTERNET_TASK_COLUMNS,
    dtype=INTERNET_TASK_DTYPES,
):
    opt_rows += len(chunk)
    opt_total_sum += float(chunk["internet_traffic"].sum(skipna=True))
    opt_total_39 += float(
        chunk.loc[chunk["country_code"] == 39, "internet_traffic"].sum(skipna=True)
    )
    chunk_aggs_sum.append(aggregate_internet_traffic(chunk, strategy="sum_all_countries"))
    chunk_aggs_39.append(aggregate_internet_traffic(chunk, strategy="country_39_only"))
    peak_opt = max(peak_opt, get_process_rss_mb())
    del chunk

agg_sum_opt = add_timestamp_column(merge_chunk_aggregates(chunk_aggs_sum))
agg_39_opt = add_timestamp_column(merge_chunk_aggregates(chunk_aggs_39))
agg_sum_opt = agg_sum_opt.sort_values(["square_id", "time_interval"]).reset_index(drop=True)
agg_39_opt = agg_39_opt.sort_values(["square_id", "time_interval"]).reset_index(drop=True)
opt_seconds = time.perf_counter() - t0
agg_sum_mem_mb = agg_sum_opt.memory_usage(deep=True).sum() / (1024 * 1024)

print("Optimized seconds:", round(opt_seconds, 3))
print("Rows read:", opt_rows)
print("RSS before (MB):", round(mem_before_opt, 2))
print("RSS peak (MB):", round(peak_opt, 2))
print("RSS peak delta (MB):", round(peak_opt - mem_before_opt, 2))
print("Aggregated sum_all DataFrame memory (MB):", round(agg_sum_mem_mb, 2))
print(agg_sum_opt.head())

## 1.6 Memory Comparison

In [ ]:
baseline_rss_delta = mem_after_baseline - mem_before_baseline
optimized_rss_peak_delta = peak_opt - mem_before_opt

df_reduction_pct = (df_base_mem_mb - agg_sum_mem_mb) / df_base_mem_mb * 100
rss_reduction_pct = (baseline_rss_delta - optimized_rss_peak_delta) / baseline_rss_delta * 100

comparison = pd.DataFrame(
    [
        {
            "metric": "Process RSS delta (MB)",
            "baseline": baseline_rss_delta,
            "optimized": optimized_rss_peak_delta,
            "reduction_pct": rss_reduction_pct,
        },
        {
            "metric": "Retained DataFrame memory (MB)",
            "baseline": df_base_mem_mb,
            "optimized": agg_sum_mem_mb,
            "reduction_pct": df_reduction_pct,
        },
    ]
)
display(comparison)
print(
    "Note: DataFrame comparison is full raw day table vs aggregated Internet table; "
    "RSS comparison is baseline load delta vs optimized peak delta during chunked processing."
)

## 1.7 Internet Traffic Aggregation

Compare `sum_all_countries` vs `country_39_only`, then save the recommended day-level sample.

In [ ]:
strategy_comparison = pd.DataFrame(
    [
        {
            "strategy": "sum_all_countries",
            "n_rows": len(agg_sum_opt),
            "internet_sum": float(agg_sum_opt["internet_traffic"].sum(skipna=True)),
        },
        {
            "strategy": "country_39_only",
            "n_rows": len(agg_39_opt),
            "internet_sum": float(agg_39_opt["internet_traffic"].sum(skipna=True)),
        },
    ]
)
strategy_comparison["share_of_sum_all"] = (
    strategy_comparison["internet_sum"] / strategy_comparison.loc[0, "internet_sum"]
)
display(strategy_comparison)

RECOMMENDED_STRATEGY = "sum_all_countries"
sample_path = SAMPLES_DIR / "processed_sample_day.csv"
sample_out = agg_sum_opt[["timestamp", "square_id", "internet_traffic"]].copy()
sample_out.to_csv(sample_path, index=False)
print("Wrote", sample_path, "rows=", len(sample_out), "strategy=", RECOMMENDED_STRATEGY)

## 1.8 Correctness Validation

In [ ]:
merged = agg_sum_base.merge(
    agg_sum_opt[["square_id", "time_interval", "internet_traffic"]],
    on=["square_id", "time_interval"],
    how="outer",
    suffixes=("_base", "_opt"),
)
abs_diff = (
    merged["internet_traffic_base"].astype("float64")
    - merged["internet_traffic_opt"].astype("float64")
).abs()

validation = {
    "row_count_match": n_rows == opt_rows,
    "agg_sum_rows_match": len(agg_sum_base) == len(agg_sum_opt),
    "agg_39_rows_match": len(agg_39_base) == len(agg_39_opt),
    "total_internet_abs_diff": abs(base_total_sum - opt_total_sum),
    "total_internet_rel_diff": abs(base_total_sum - opt_total_sum) / abs(base_total_sum),
    "agg_sum_max_abs_diff": float(abs_diff.max()),
    "agg_sum_n_mismatched_keys": int(
        (merged["internet_traffic_base"].isna() ^ merged["internet_traffic_opt"].isna()).sum()
    ),
    "timestamps_sorted_within_square": bool(
        agg_sum_opt.groupby("square_id")["time_interval"].apply(lambda s: s.is_monotonic_increasing).all()
    ),
}

spot_checks = []
for sid, tms, expected in spot_keys:
    row = agg_sum_opt[(agg_sum_opt["square_id"] == sid) & (agg_sum_opt["time_interval"] == tms)]
    got = float(row["internet_traffic"].iloc[0]) if len(row) else None
    spot_checks.append(
        {
            "square_id": int(sid),
            "time_interval": int(tms),
            "baseline": float(expected),
            "optimized": got,
            "abs_diff": abs(float(expected) - got) if got is not None else None,
        }
    )

print(json.dumps(validation, indent=2))
display(pd.DataFrame(spot_checks))

## 1.9 Phase 2 Findings

In [ ]:
remove_path(txt_path)
print("Temporary extracted TXT deleted:", not txt_path.exists())

findings = {
    "representative_file": REPRESENTATIVE_MEMBER,
    "chunksize": CHUNKSIZE,
    "raw_row_count": n_rows,
    "n_unique_squares": len(unique_squares),
    "n_unique_timestamps": len(unique_ts),
    "recommended_country_aggregation": RECOMMENDED_STRATEGY,
    "internet_share_country_39": float(
        cc_summary.loc[cc_summary["country_code"] == 39, "internet_share"].sum()
    ),
    "baseline_dataframe_mb": float(df_base_mem_mb),
    "optimized_agg_dataframe_mb": float(agg_sum_mem_mb),
    "dataframe_memory_reduction_pct": float(df_reduction_pct),
    "rss_delta_reduction_pct": float(rss_reduction_pct),
    "validation_row_count_match": validation["row_count_match"],
    "validation_max_abs_diff": validation["agg_sum_max_abs_diff"],
}
print(json.dumps(findings, indent=2))
print("\nPhase 2 complete. Do not proceed to EDA/full-dataset processing until reviewed.")

# 2. Phase 3A — Full-Dataset Processing

Process all **62** daily files sequentially (one TXT at a time, chunked reads) to build compact summaries for Phase 3B area selection.

**Outputs:**
- `data/processed/square_internet_totals.csv`
- `data/processed/daily_square_internet_totals.csv`
- `results/metrics/phase3_dataset_summary.json`
- `results/metrics/phase3_processing_log.csv`

Aggregation method: **`sum_all_countries`**. Missing square×timestamp cells are **not** filled with zeros.


## 2.1 Run full processing pipeline

Set `DATA_DIR` / `MILAN_DATA_DIR` to the folder containing the Dataverse ZIP archives. Skip re-running if outputs already exist and you only want to inspect results.


In [ ]:
from pathlib import Path
import json
import os
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

from src.data.preprocessing import build_square_traffic_summary

DATA_DIR = Path(os.environ.get("MILAN_DATA_DIR") or os.environ.get("DATA_DIR") or r"C:\Users\user\Downloads")
SUMMARY_PATH = ROOT / "results" / "metrics" / "phase3_dataset_summary.json"

RUN_FULL_PIPELINE = not SUMMARY_PATH.exists()  # set True to force recompute

if RUN_FULL_PIPELINE:
    summary = build_square_traffic_summary(
        data_dir=DATA_DIR,
        chunksize=500_000,
        strategy="sum_all_countries",
    )
else:
    summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
    print("Loaded existing Phase 3A summary from", SUMMARY_PATH)

print("Files processed:", summary["number_of_daily_files"], "/", summary["number_of_daily_files_discovered"])
print("Date range:", summary["first_date"], "->", summary["last_date"])
print("Processing time (seconds):", summary["processing_time_seconds"])
print("Peak RSS (MB):", summary["peak_memory_mb"])


## 2.2 Dataset summary and top squares


In [ ]:
display(pd.Series({
    "zip_files": summary["number_of_zip_files"],
    "daily_files": summary["number_of_daily_files"],
    "first_date": summary["first_date"],
    "last_date": summary["last_date"],
    "unique_squares": summary["number_of_unique_squares"],
    "raw_rows": summary["total_raw_rows_processed"],
    "aggregated_square_time_rows": summary["total_aggregated_square_time_rows"],
    "unique_timestamps": summary["number_of_unique_timestamps"],
    "country_codes": summary["number_of_country_codes"],
    "total_internet_traffic": summary["total_internet_traffic"],
    "missing_or_invalid_rows": summary["missing_or_invalid_rows"],
    "aggregation_method": summary["aggregation_method"],
}))

top10 = pd.DataFrame(summary["top_10_squares_by_total_internet_traffic"])
display(top10)
print("Top 3 square_ids:", [r["square_id"] for r in summary["top_3_squares_by_total_internet_traffic"]])


## 2.3 Validation


In [ ]:
val = summary["validation"]
validation_view = pd.Series({
    "exact_62_files": val.get("exact_62_files"),
    "dates_continuous": val.get("dates_continuous"),
    "duplicate_dates": val.get("duplicate_dates"),
    "missing_dates": val.get("missing_dates"),
    "all_files_processed": val.get("all_files_processed"),
    "incomplete": val.get("incomplete"),
    "cumulative_equals_daily_sum": val.get("cumulative_equals_daily_sum"),
    "abs_diff_daily_vs_square_sum": val.get("abs_diff_daily_vs_square_sum"),
    "zip_open_failures": val.get("zip_open_failures"),
    "file_failures": val.get("file_failures"),
    "fraction_global_diffs_10_minutes": val.get("fraction_global_diffs_10_minutes"),
})
display(validation_view)

log = pd.read_csv(ROOT / "results" / "metrics" / "phase3_processing_log.csv")
print("Processing log rows:", len(log))
print("All days schema_ok:", bool(log["schema_ok"].all()))
print("All days 10-minute timestamps:", bool(log["timestamps_all_10_minutes"].all()))
display(log.head())


## 2.4 Phase 3A findings

Phase 3A stops here. Compact square totals and daily totals are ready for Phase 3B EDA / targeted extraction. Do **not** start EDA or model selection in this section.


# 3. PHASE 3B — Exploratory Data Analysis

Targeted 10-minute Internet series for five squares, then EDA for forecasting design.

**Target squares**
- Top 3 by total Internet (Phase 3A): `5161`, `5059`, `5259`
- Assignment-required: `4159`, `4556`

Aggregation remains **`sum_all_countries`**. Missing bins are measured, not zero-filled.


## 3.1 Targeted extraction (or load existing output)


In [ ]:
from pathlib import Path
import json
import os
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

from src.data.preprocessing import TARGET_SQUARES, TOP_SQUARES, extract_target_square_timeseries
from src.analysis.eda import (
    analyse_missingness,
    descriptive_statistics,
    first_two_weeks_mask,
    load_square_totals,
    load_target_timeseries,
    save_daily_target_plot,
    save_first_two_weeks_plots,
    save_missingness_figure,
    save_square_acf,
    save_square_seasonality,
    save_top_areas_bar,
    save_traffic_distribution,
    save_unusual_period_plot,
)

DATA_DIR = Path(os.environ.get("MILAN_DATA_DIR") or os.environ.get("DATA_DIR") or r"C:\Users\user\Downloads")
PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "results" / "figures"
METRICS = ROOT / "results" / "metrics"
TS_PATH = PROCESSED / "target_square_internet_timeseries.csv"

print("TOP_SQUARES:", TOP_SQUARES)
print("TARGET_SQUARES:", TARGET_SQUARES)

if not TS_PATH.exists():
    extract_target_square_timeseries(data_dir=DATA_DIR, target_squares=TARGET_SQUARES)
else:
    print("Using existing", TS_PATH)

ts = load_target_timeseries(TS_PATH)
totals = load_square_totals(PROCESSED / "square_internet_totals.csv")
print(ts.groupby("square_id").size())
print("timestamp range:", ts["timestamp"].min(), "->", ts["timestamp"].max())
assert sorted(ts["square_id"].unique()) == sorted(TARGET_SQUARES)
assert totals.head(3)["square_id"].astype(int).tolist() == TOP_SQUARES


## 3.2 Missingness (no zero-filling)


In [ ]:
missingness = analyse_missingness(ts, target_squares=TARGET_SQUARES)
(METRICS / "phase3b_missingness.json").write_text(json.dumps(missingness, indent=2), encoding="utf-8")
miss_df = pd.DataFrame(missingness["by_square"].values())
display(miss_df)
save_missingness_figure(missingness, FIGURES / "eda_target_squares_missingness.png")
display(Image(filename=str(FIGURES / "eda_target_squares_missingness.png")))


## 3.3 EDA 1 — Traffic distribution across all 10,000 squares


In [ ]:
save_traffic_distribution(totals, FIGURES / "eda_traffic_distribution.png", top_squares=TOP_SQUARES)
display(Image(filename=str(FIGURES / "eda_traffic_distribution.png")))
display(totals.head(10))
print("TOP 3:", TOP_SQUARES)


## 3.4 EDA 2 — Top areas ranking


In [ ]:
save_top_areas_bar(totals, FIGURES / "eda_top_areas.png", n=15, top_squares=TOP_SQUARES)
display(Image(filename=str(FIGURES / "eda_top_areas.png")))


## 3.5 EDA 3 — First two weeks (2013-11-01 through 2013-11-14)


In [ ]:
first2 = ts.loc[first_two_weeks_mask(ts)]
print("First-two-weeks rows:", len(first2), "dates:", sorted(first2["date"].unique())[:3], "...", sorted(first2["date"].unique())[-3:])
paths = save_first_two_weeks_plots(ts, FIGURES, target_squares=TARGET_SQUARES)
for p in paths:
    display(Image(filename=str(p)))


## 3.6 EDA 4 — Descriptive statistics / temporal dynamics


In [ ]:
stats = descriptive_statistics(ts, target_squares=TARGET_SQUARES, missingness=missingness)
stats.to_csv(METRICS / "target_square_descriptive_statistics.csv", index=False)
display(stats.sort_values("mean", ascending=False))
print("Highest mean:", int(stats.loc[stats["mean"].idxmax(), "square_id"]))
print("Highest CV (most variable relative to mean):", int(stats.loc[stats["coefficient_of_variation"].idxmax(), "square_id"]))
print("Lowest CV (most stable relative to mean):", int(stats.loc[stats["coefficient_of_variation"].idxmin(), "square_id"]))


## 3.7 EDA 5 — Daily totals across the full period


In [ ]:
daily = save_daily_target_plot(ts, FIGURES / "eda_target_squares_daily.png", target_squares=TARGET_SQUARES)
display(Image(filename=str(FIGURES / "eda_target_squares_daily.png")))
display(daily.head())


## 3.8 EDA 6 — Square 5161 deep analysis

### A. Autocorrelation (required)
### B. Seasonality (hour-of-day / day-of-week) — chosen because daily/weekly structure is visible in the series


In [ ]:
acf_info = save_square_acf(ts, FIGURES / "eda_square_5161_acf.png", square_id=5161, nlags=1100)
display(Image(filename=str(FIGURES / "eda_square_5161_acf.png")))
print(json.dumps(acf_info["acf_at_key_lags"], indent=2))

seasonality = save_square_seasonality(ts, FIGURES / "eda_square_5161_seasonality.png", square_id=5161)
display(Image(filename=str(FIGURES / "eda_square_5161_seasonality.png")))
print("peak hour (Europe/Rome):", seasonality["peak_hour_local"])
print("low hour:", seasonality["low_hour_local"])
print("peak weekday:", seasonality["peak_weekday"], "low weekday:", seasonality["low_weekday"])


## 3.9 EDA 7 — Unusual period in Square 5161


In [ ]:
unusual = save_unusual_period_plot(ts, FIGURES / "eda_square_5161_unusual_period.png", square_id=5161)
display(Image(filename=str(FIGURES / "eda_square_5161_unusual_period.png")))
print(json.dumps(unusual, indent=2))


## 3.10 EDA Findings Relevant to Forecasting

Evidence-based notes (not final model selection):

1. **Traffic distribution:** Highly right-skewed across 10,000 squares; a small set of cells carries much higher totals (top 3: 5161, 5059, 5259).
2. **Area differences:** Means and CVs differ substantially — 5161 has the highest mean and high relative volatility; 4556 is lower-mean and comparatively more stable (lowest CV).
3. **Temporal dependence:** ACF for 5161 is very strong at short lags (~0.99 at 10 min, ~0.94 at 1 h) and remains high at 1 day (~0.88) and 1 week (~0.84), with a negative trough near 12 h — consistent with day/night antipodal structure.
4. **Seasonality:** Clear hour-of-day and weekday profiles (Europe/Rome) support models that can represent daily/weekly periodic structure or features.
5. **Variability / anomalies:** Occasional extreme spikes (e.g. robust-z outlier around 2013-12-27 15:30 UTC in 5161) create forecasting risk for error metrics sensitive to peaks.
6. **Missingness:** For these five squares, missing 10-minute bins vs the full grid are **0%** in the extracted series — favourable for supervised sequence construction without immediate zero-imputation.
7. **Smooth vs volatile:** Series are not white noise; they are strongly autocorrelated but still spiky at peaks, especially in high-traffic cells.

### Model-characteristic shortlist (not final three models)

Based on the EDA, later model selection should favour approaches that can:

- capture **strong short-term autocorrelation** (lag-1 / within-hour persistence);
- represent **daily and weekly periodicity** (seasonal structure / seasonal features / periodic components);
- handle **nonlinear or regime-like peak behaviour** in high-traffic squares;
- remain competitive on **volatile, right-skewed** targets (possibly via transforms or robust training);
- include at least one **interpretable statistical baseline** that uses autocorrelation/seasonality explicitly.

Final choice of three models is deferred to the next phase after review.


# 4. Phase 5 — Forecasting Experiment Design & Data Preparation

Prepare the one-step-ahead experiment for the **locked** models **SARIMA**, **LSTM**, and **TCN**.

This section does **not** train those models, does not tune hyperparameters, and does not produce a final comparison table.

It uses the existing Phase 3B file `data/processed/target_square_internet_timeseries.csv` rather than reprocessing raw ZIPs.


## 5.1 Selected forecasting areas

Primary forecasting squares (Phase 3A highest total Internet traffic):

- **5161**
- **5059**
- **5259**

Squares **4159** and **4556** were required for EDA. They are **not** used as silent substitutes in the forecasting experiment.

Target variable: `internet_traffic`. Resolution: **10 minutes**.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config as cfg
from src.data.forecasting import (
    alignment_example,
    analyse_sequence_lengths,
    assess_sarima_seasonal_period,
    build_forecasting_dataset,
    fit_square_scalers,
    inspect_zero_traffic,
    load_source_target_timeseries,
    make_supervised_sequences,
    naive_persistence_forecast,
    prepare_phase5_experiment,
    univariate_series,
    validate_forecasting_dataset,
)
from src.evaluation.metrics import mae, mape, mape_diagnostics, rmse

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

print("Locked research models:", cfg.FORECASTING_MODELS)
print("Baseline (not a research model):", cfg.NAIVE_BASELINE_NAME)
print("TARGET_SQUARES:", cfg.TARGET_SQUARES)
print("TARGET_COLUMN:", cfg.TARGET_COLUMN)
print("FREQUENCY:", cfg.FREQUENCY)
print("EDA squares (not all used for forecasting):", cfg.EDA_SQUARES)


## 5.2 Forecasting definition

One-step-ahead forecasting:

Given traffic observations available through time \( t \), predict Internet traffic at \( t + 10 \) minutes.

\[
\hat{y}_{t+1} = f(y_{\leq t})
\]

One step = **10 minutes**. The target value, and any later observation, must not enter the forecast issued at time \( t \).


## 5.3 Chronological split

Random splitting is **not** used. Shuffling a serially correlated 10-minute series would let future timestamps influence training, scaler fit, and sequence windows.

Final test period (unseen during fitting and configuration selection):

- **2013-12-16 00:00 UTC** through **2013-12-22 23:50 UTC**
- \( 7 	imes 144 = 1008 \) observations per square

| Split | UTC start | UTC end | Rows / square |
|-------|-----------|---------|---------------|
| train | 2013-11-01 00:00 | 2013-12-08 23:50 | 5472 |
| validation | 2013-12-09 00:00 | 2013-12-15 23:50 | 1008 |
| test | 2013-12-16 00:00 | 2013-12-22 23:50 | 1008 |

Bounds are applied to the UTC `timestamp` column. Rows outside these windows are excluded, not zero-filled.


In [ ]:
phase5 = prepare_phase5_experiment(root=ROOT, write_outputs=True)
forecast_df = phase5["dataset"]

print("Forecasting dataset rows:", len(forecast_df))
print("Columns:", list(forecast_df.columns))
print("Squares:", sorted(forecast_df["square_id"].unique().tolist()))
print("\nRows by square and split:")
print(
    forecast_df.groupby(["square_id", "split"], observed=True)
    .size()
    .unstack("split")
)
print("\nChronological range:", forecast_df["timestamp"].min(), "→", forecast_df["timestamp"].max())
print("Monotonic per square:",
      all(g["timestamp"].is_monotonic_increasing for _, g in forecast_df.groupby("square_id")))
print("Unused Phase 3B rows (outside the three splits):", phase5["n_unused_source_rows"])
print("models_trained:", phase5["models_trained"])


## 5.4 Dataset integrity checks

Checks (per square): 10-minute frequency, no duplicate or missing timestamps, no unexpected square IDs, chronological order, expected row counts, and no test leakage into train or validation.

If a check fails, preparation stops. Missing timestamps are **not** interpolated or replaced with zeros.


In [ ]:
validation = phase5["validation"]
display(validation)

issues = list(validation.attrs.get("issues", []))
print("Integrity issues:", issues if issues else "none")

assert set(forecast_df["square_id"].unique()) == set(cfg.TARGET_SQUARES)
for sid in cfg.TARGET_SQUARES:
    sub = forecast_df.loc[forecast_df["square_id"] == sid]
    assert int((sub["split"] == "train").sum()) == 5472
    assert int((sub["split"] == "validation").sum()) == 1008
    assert int((sub["split"] == "test").sum()) == 1008
    test_start, test_end = cfg.utc_timestamp(cfg.TEST_START), cfg.utc_timestamp(cfg.TEST_END)
    train = sub.loc[sub["split"] == "train", "timestamp"]
    val = sub.loc[sub["split"] == "validation", "timestamp"]
    assert not ((train >= test_start) & (train <= test_end)).any()
    assert not ((val >= test_start) & (val <= test_end)).any()
    assert sub["timestamp"].is_monotonic_increasing
    assert sub["timestamp"].duplicated().sum() == 0
print("Leakage and count assertions passed for squares", cfg.TARGET_SQUARES)


## 5.5 Sequence-length investigation (LSTM / TCN)

Supervised form (no training in this cell):

- \( X = [y_{t-L+1},\ldots,y_t] \)
- target \( = y_{t+1} \)

Candidates: \( L=144 \) (24 h), \( L=288 \) (48 h), \( L=1008 \) (7 days).

Phase 3B ACF (square 5161): lag 1 ≈ 0.99, lag 144 ≈ 0.88, lag 1008 ≈ 0.84.

**Selected primary length: 144**, used for **both** LSTM and TCN. It includes the daily seasonal counterpart of the target, matches the one-step-ahead objective, and avoids a 7× LSTM unroll. Weekly lag 1008 is documented as out of window; it is not a silent omission.


In [ ]:
seq_tbl = phase5["sequence_length_analysis"]
display(seq_tbl)

print("Primary sequence length:", cfg.PRIMARY_SEQUENCE_LENGTH)
print("Same length for LSTM and TCN:", True)

# Leakage: training windows must not include validation/test observations
seq_train = make_supervised_sequences(
    forecast_df, square_id=5161, sequence_length=cfg.PRIMARY_SEQUENCE_LENGTH, target_split="train"
)
print("Square 5161 train sequences:", seq_train["n_samples"], "X shape", seq_train["X"].shape)
assert seq_train["target_timestamp"].max() <= cfg.utc_timestamp(cfg.TRAIN_END)

seq_test = make_supervised_sequences(
    forecast_df, square_id=5161, sequence_length=cfg.PRIMARY_SEQUENCE_LENGTH, target_split="test"
)
print("Square 5161 test sequences:", seq_test["n_samples"])
assert seq_test["n_samples"] == 1008
print("First test target:", seq_test["target_timestamp"][0])
print("First test input end:", seq_test["input_end_timestamp"][0])


## 5.6 Normalization strategy

LSTM/TCN scaler: **per square**, **MinMax** to \([0, 1]\), **fitted on training data only**, then applied to train, validation, and test.

This prevents leakage of validation/test extrema into the scale. Predictions must be **inverse-transformed** before MAE, RMSE, and MAPE. SARIMA does not use this scaler. The CSV stores original-scale traffic.


In [ ]:
scalers = phase5["scalers"]
print(json.dumps(phase5["scaler_params"], indent=2))

for sid, scaler in scalers.items():
    train_vals = forecast_df.loc[
        (forecast_df["square_id"] == sid) & (forecast_df["split"] == "train"), "internet_traffic"
    ].to_numpy()
    assert int(scaler.scaler.n_samples_seen_) == len(train_vals)
    assert np.isclose(scaler.scaler.data_min_[0], train_vals.min())
    assert np.isclose(scaler.scaler.data_max_[0], train_vals.max())
    scaled = scaler.transform(train_vals)
    recovered = scaler.inverse_transform(scaled)
    assert np.allclose(recovered, train_vals)
print("Train-only scaler fit verified; inverse-transform round-trip OK.")


## 5.7 SARIMA input design

SARIMA receives a chronological univariate 10-minute series (training split for fitting). It does **not** use the neural sliding-window matrix.

Seasonal period:

- \( s=144 \) is statistically justified (daily ACF) and is the **recommended** period.
- \( s=1008 \) is statistically visible but impractical in a single SARIMA (thin weekly sample in 38 training days; heavy seasonal lag; seasonal differencing of order 1008).
- Daily and weekly periods are **not** combined in one SARIMA specification.

No SARIMA model is fitted in this phase.


In [ ]:
sarima_note = assess_sarima_seasonal_period()
print(json.dumps(sarima_note, indent=2))

s5161_train = univariate_series(forecast_df, 5161, split="train")
print("SARIMA training series 5161:", len(s5161_train), "freq", s5161_train.index.freqstr)
print(s5161_train.head(3))
print("...")
print(s5161_train.tail(2))
assert str(s5161_train.index.freqstr) in {"10min", "10T"}
assert s5161_train.isna().sum() == 0
print("SARIMA training series is a complete 10-minute univariate index. Model not fitted.")


## 5.8 Naive persistence baseline

**Naive persistence baseline** (not one of the three research models):

\[
\hat{y}_{t+1} = y_t
\]

It is a reference for whether SARIMA, LSTM, and TCN improve on last-value persistence.


In [ ]:
align = phase5["alignment_example"]
display(align)

print("Required alignment:")
print("  Input ends: 2013-12-15 23:50")
print("  Target:     2013-12-16 00:00")

for _, row in align.iterrows():
    assert str(row["input_end_timestamp"]).startswith("2013-12-15 23:50")
    assert str(row["target_timestamp"]).startswith("2013-12-16 00:00")
    assert int(row["step_minutes"]) == 10
    assert row["target_split"] == "test"
    assert np.isclose(row["naive_persistence_prediction"], row["input_last_value"])

series = univariate_series(forecast_df, 5161)
naive = naive_persistence_forecast(series)
target_ts = cfg.utc_timestamp("2013-12-16 00:00")
print("Persistence ŷ(2013-12-16 00:00) = y(2013-12-15 23:50) =", float(naive.loc[target_ts]))
print("Actual y(2013-12-16 00:00) =", float(series.loc[target_ts]))
print("No research model trained.")


## 5.9 Metric definitions

- **MAE** and **RMSE**: all observations; original traffic scale.
- **MAPE**: denominator \(\max(|y_t|, 10^{-8})\). Observations are **not** dropped when \( y_t \) is zero or tiny.

On these three squares, train/validation/test traffic has **no zeros** and minima well above 1, so the floor is a safeguard rather than an active correction.


In [ ]:
zeros = phase5["zero_inspection"]
display(zeros)

y_demo = np.array([100.0, 0.0, 50.0])
yhat_demo = np.array([110.0, 5.0, 40.0])
print("Demo MAE (all 3 points):", mae(y_demo, yhat_demo))
print("Demo RMSE (all 3 points):", rmse(y_demo, yhat_demo))
print("Demo MAPE with floor, no rows dropped:", mape(y_demo, yhat_demo))
print(mape_diagnostics(y_demo))
print("MAPE floor:", cfg.MAPE_DENOMINATOR_FLOOR)
print("These demo numbers are not model results.")


## 5.10 Final experiment configuration

All of the following are defined once in `src/config.py`:

- `TARGET_SQUARES = [5161, 5059, 5259]`
- `TARGET_COLUMN = "internet_traffic"`
- `FREQUENCY = "10min"`
- `TEST_START / TEST_END`
- `VALIDATION_START / VALIDATION_END`
- plus matching train bounds, \( L=144 \), and MAPE floor

Phase 5 stops here. **Do not train SARIMA, LSTM, or TCN until this design has been reviewed.**


In [ ]:
print(json.dumps(phase5["config"], indent=2))
print("Wrote:")
for key, path in phase5["outputs"].items():
    print(f"  {key}: {path}")
print("\nConfirmation: models trained in Phase 5 =", phase5["models_trained"])
assert phase5["models_trained"] is False
print("Phase 5 complete. No SARIMA, LSTM, or TCN fitting was performed.")


# 5. Phase 6A — Model Implementation

Implement the locked research models **SARIMA**, **LSTM**, and **TCN**, plus the **Naive persistence** reference baseline.

This section:

- shows architectures, shapes, and default configurations
- runs **smoke tests** on small train/validation subsets
- records fit/predict timing hooks

It does **not**:

- evaluate the reserved test week (2013-12-16 through 2013-12-22)
- tune hyperparameters on the test set
- claim that any model is better


## 6A.1 Setup and Phase 5 artefacts

Reuse the Phase 5 forecasting dataset and train-only scalers. Do not reprocess raw ZIPs.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config as cfg
from src.data.forecasting import (
    alignment_example,
    fit_square_scalers,
    make_supervised_sequences,
    prepare_phase5_experiment,
    univariate_series,
)
from src.models import (
    LSTMConfig,
    LSTMModel,
    NaivePersistenceModel,
    SARIMAConfig,
    SARIMAModel,
    TCNConfig,
    TCNModel,
)
from src.models.base import assert_boundary_alignment_example, reshape_sequences_for_keras

forecast_path = ROOT / cfg.FORECASTING_DATASET_FILE
if forecast_path.is_file():
    forecast_df = pd.read_csv(forecast_path, parse_dates=["timestamp"])
    if forecast_df["timestamp"].dt.tz is None:
        forecast_df["timestamp"] = forecast_df["timestamp"].dt.tz_localize("UTC")
else:
    forecast_df = prepare_phase5_experiment(root=ROOT, write_outputs=True)["dataset"]

scalers = fit_square_scalers(forecast_df, squares=cfg.TARGET_SQUARES)
print("Squares:", cfg.TARGET_SQUARES)
print("L =", cfg.PRIMARY_SEQUENCE_LENGTH)
print("Research models:", cfg.FORECASTING_MODELS)
print("Baseline:", cfg.NAIVE_BASELINE_NAME)
print("Rows:", len(forecast_df))


## 6A.2 Sequence alignment check

A window ending at **2013-12-15 23:50** must target **2013-12-16 00:00**. This verifies the Phase 5 sequence builder; it does not evaluate test-set accuracy.


In [ ]:
align = alignment_example(forecast_df, square_id=5161, sequence_length=cfg.PRIMARY_SEQUENCE_LENGTH)
display(align)
assert_boundary_alignment_example(align.iloc[0]["input_end_timestamp"], align.iloc[0]["target_timestamp"])
print("Boundary alignment OK. No test metrics computed.")


## 6A.3 Model configurations

Defaults are centralised in `src/config.py`. SARIMA uses unscaled traffic and \(s=144\). LSTM and TCN share \(L=144\) and the train-only MinMax scaler.


In [ ]:
print("=== SARIMA ===")
print(json.dumps(SARIMAConfig().to_dict(), indent=2))
print("\n=== LSTM ===")
print(json.dumps(LSTMConfig().to_dict(), indent=2))
print("\n=== TCN ===")
print(json.dumps(TCNConfig().to_dict(), indent=2))
print("\n=== Naive ===")
print(json.dumps(NaivePersistenceModel().get_config(), indent=2))


## 6A.4 Architecture summaries and input/output shapes


In [ ]:
lstm_demo = LSTMModel(LSTMConfig(), scaler=scalers[5161])
lstm_demo.build()
print(lstm_demo.summary_text())
print("LSTM shapes:", lstm_demo.input_output_shapes())

tcn_demo = TCNModel(TCNConfig(), scaler=scalers[5161])
tcn_demo.build()
print(tcn_demo.summary_text())
print("TCN shapes:", tcn_demo.input_output_shapes())
print("TCN receptive field:", tcn_demo.config.receptive_field())


## 6A.5 Smoke tests (small subsets only)

Fit each model on a **small training subset**. Validation may be used for neural early stopping. The reserved Dec 16–22 test week is **not** used for fitting, early stopping, or metrics.


In [ ]:
# Prefer the scripted smoke suite for a complete check; also demonstrate here.
import importlib.util
_smoke_path = ROOT / "scripts" / "run_phase6a_smoke_tests.py"
_spec = importlib.util.spec_from_file_location("phase6a_smoke", _smoke_path)
_smoke = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_smoke)
smoke_lstm = _smoke.smoke_lstm
smoke_naive = _smoke.smoke_naive
smoke_sarima = _smoke.smoke_sarima
smoke_tcn = _smoke.smoke_tcn
test_alignment = _smoke.test_alignment

align_report = test_alignment(forecast_df)
naive_report = smoke_naive(forecast_df)
print("Naive OK:", naive_report["ok"], "timing:", naive_report["timing"])

print("Running SARIMA smoke (short subset, s=144)...")
sarima_report = smoke_sarima(forecast_df)
print("SARIMA OK:", sarima_report["ok"], "fit_s:", sarima_report["timing"]["fit_seconds"])

print("Running LSTM smoke...")
lstm_report = smoke_lstm(forecast_df, scalers[5161])
print(
    "LSTM OK:", lstm_report["ok"],
    "in", lstm_report["keras_input_shape"],
    "out", lstm_report["keras_output_shape"],
    "fit_s:", lstm_report["timing"]["fit_seconds"],
)

print("Running TCN smoke...")
tcn_report = smoke_tcn(forecast_df, scalers[5161])
print(
    "TCN OK:", tcn_report["ok"],
    "in", tcn_report["keras_input_shape"],
    "out", tcn_report["keras_output_shape"],
    "fit_s:", tcn_report["timing"]["fit_seconds"],
)

print("\nConfirmation:")
print("  Models implemented: SARIMA, LSTM, TCN + Naive persistence")
print("  Final Dec 16–22 evaluation performed: False")
print("  Performance ranking claimed: False")


## 6A.6 Timing implementation

Each model records wall-clock `fit_seconds` and `predict_seconds` via `src.models.base.Timer`. Neural models also store epochs completed, train/validation loss, and best epoch when early stopping is used. Early stopping never monitors the test set.

Phase 6A stops here. Do **not** proceed to full training or Dec 16–22 evaluation until reviewed.
